# BERTopic — Influencer Topic Modelling

Train a BERTopic model on Instagram influencer bios to automatically
categorise influencers by topic (fitness, food, travel, beauty …).

Pipeline: **all-MiniLM-L6-v2 embeddings → UMAP → HDBSCAN → c-TF-IDF**

## 1. Imports

In [ ]:
from bertopic import BERTopic
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

from services.influencers_service import InfluencersService
from shared.constants import NOISE_WORDS
from shared.mongo import MongoConnection

## 2. Load training data

Build a list of documents from influencer **bios + categories**,
stripping noise words that leak into every profile.

In [ ]:
mongo_connection = MongoConnection()
influencers_service = InfluencersService(mongo_connection)

bios = influencers_service.get_bio_documents(noise_words=NOISE_WORDS)

print(f"{len(bios)} documents loaded")
bios[:5]

## 3. Create & fit the model

Use `all-MiniLM-L6-v2` for sentence embeddings.  
Default UMAP + HDBSCAN for dimensionality reduction and clustering.

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

topic_model = BERTopic(
    embedding_model=EMBEDDING_MODEL,
    verbose=True,
)

topics, probs = topic_model.fit_transform(bios)
print(f"\nGenerated {len(set(topics)) - 1} topics (excluding outlier topic -1)")

## 4. Inspect topics

In [ ]:
topic_model.get_topic_info().head(15)

In [ ]:
# Look at the top words for a specific topic
topic_model.get_topic(topic=0)

In [ ]:
# Representative documents for a topic
topic_model.get_representative_docs(0)

## 5. Visualise

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart()

## 6. Fine-tune topics

Three knobs to turn:
1. **Reduce** the total number of topics.
2. **Merge** semantically similar topics (e.g. country-related clusters → *travel*).
3. **Update** the vectorizer to use bigrams/trigrams for richer labels.

In [ ]:
# 6a. Reduce to ~100 topics for broader categories
topic_model.reduce_topics(bios, nr_topics=100)
print(f"After reduction: {len(set(topic_model.topics_)) - 1} topics")

In [ ]:
# 6b. Merge country/city clusters into a single "travel" topic
# Adjust the topic IDs after inspecting visualize_topics()
# topic_model.merge_topics(bios, [0, 4, 5])  # example IDs

In [ ]:
# 6c. Use n-grams for better topic labels
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words="english")
topic_model.update_topics(bios, vectorizer_model=vectorizer_model)

In [ ]:
# Re-inspect after fine-tuning
topic_model.visualize_barchart()

## 7. Coherence score (c_v)

Quantitative evaluation via Gensim's `CoherenceModel`.

In [ ]:
documents = pd.DataFrame(
    {"Document": bios, "ID": range(len(bios)), "Topic": topic_model.topics_}
)
documents_per_topic = documents.groupby(["Topic"], as_index=False).agg(
    {"Document": " ".join}
)
cleaned_docs = topic_model._preprocess_text(documents_per_topic.Document.values)

vectorizer = topic_model.vectorizer_model
analyzer = vectorizer.build_analyzer()

words = vectorizer.get_feature_names_out()
tokens = [analyzer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]

topic_words = [
    [word for word, _ in topic_model.get_topic(topic)]
    for topic in range(len(set(topic_model.topics_)) - 1)
]

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokens,
    corpus=corpus,
    dictionary=dictionary,
    coherence="c_v",
)
coherence = coherence_model.get_coherence()
print(f"Coherence score (c_v): {coherence:.4f}")

## 8. Save the model

In [ ]:
MODEL_DIR = "model/bert_9"

topic_model.save(MODEL_DIR, serialization="safetensors")
print(f"Model saved to {MODEL_DIR}")

## 9. Load & predict on new text

In [ ]:
loaded_model = BERTopic.load(MODEL_DIR, embedding_model=EMBEDDING_MODEL)

test_texts = ["sport", "vogue", "recipe healthy cooking"]
for text in test_texts:
    new_topics, new_probs = loaded_model.transform([text])
    print(f"'{text}' → topic {new_topics[0]}, prob {new_probs[0]:.4f}")